In [1]:

import pandas as pd
import numpy as np
import json
from pathlib import Path
from datetime import datetime

class DataPipeline:
    """Reusable ML data prep pipeline."""
    VERSION = '1.0.0'

    def __init__(self, name, target_column):
        self.name          = name
        self.target_column = target_column
        self.df_raw        = None
        self.df_clean      = None
        self.fill_values   = {}
        self.encodings     = {}
        self.run_log       = []

    def _log(self, msg):
        entry = {'time': datetime.now().strftime('%H:%M:%S'), 'msg': msg}
        self.run_log.append(entry)
        print(f'[{entry["time"]}] {msg}')

    def load(self, path_or_url):
        self.df_raw = pd.read_csv(path_or_url)
        self._log(f'Loaded: {self.df_raw.shape}')
        return self

    def drop_columns(self, columns):
        self.df_raw = self.df_raw.drop(columns=[c for c in columns if c in self.df_raw.columns])
        self._log(f'Dropped: {columns}'); return self

    def fill_nulls(self, strategy='median'):
        df = self.df_raw.copy()
        for col in df.select_dtypes(include='number').columns:
            if df[col].isnull().any():
                val = df[col].median() if strategy=='median' else df[col].mean()
                self.fill_values[col] = float(val); df[col] = df[col].fillna(val)
        self.df_raw = df; self._log('Nulls filled'); return self

    def encode_categoricals(self, columns):
        df = self.df_raw.copy()
        for col in columns:
            mapping = {v:i for i,v in enumerate(df[col].dropna().unique())}
            self.encodings[col] = mapping; df[col] = df[col].map(mapping)
        self.df_raw = df; self._log(f'Encoded: {columns}'); return self

    def build(self):
        self.df_clean = self.df_raw.copy()
        self._log(f'Built. Shape: {self.df_clean.shape}'); return self

    def save(self, output_dir='output'):
        out = Path(output_dir); out.mkdir(parents=True, exist_ok=True)
        self.df_clean.to_csv(out/f'{self.name}_clean.csv', index=False)
        meta = {'pipeline': self.name, 'shape': list(self.df_clean.shape),
                'fill_values': self.fill_values, 'encodings': self.encodings,
                'log': self.run_log}
        with open(out/f'{self.name}_meta.json','w') as f: json.dump(meta,f,indent=2)
        self._log(f'Saved to {out}'); return self

# Run it — method chaining like pandas/sklearn
url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
pipeline = (
    DataPipeline('titanic', 'Survived')
    .load(url)
    .drop_columns(['Cabin','Ticket','Name','PassengerId'])
    .fill_nulls()
    .encode_categoricals(['Sex','Embarked'])
    .build()
    .save('ml_project/output')
)

[17:12:49] Loaded: (891, 12)
[17:12:49] Dropped: ['Cabin', 'Ticket', 'Name', 'PassengerId']
[17:12:49] Nulls filled
[17:12:49] Encoded: ['Sex', 'Embarked']
[17:12:49] Built. Shape: (891, 8)
[17:12:49] Saved to ml_project\output
